In [ ]:
"""
6개 제품군 통합 분석
독립변수1: 친절도 측정 변수 (7가지)        → 통합 모델(제품군 더미 통제)
독립변수2: 카테고리별 주요 단어            → 제품군별 개별 회귀
종속변수 : 찜전환율=interest/view, 채팅전환율=heart/view

[수정 안내] 아래 ★수정 구역★의 리스트만 고치면 전체가 그대로 재실행됩니다.
"""
import re, numpy as np, pandas as pd, statsmodels.api as sm

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ★수정 구역 1★ : 제품군 파일 매핑  (파일명만 본인 환경에 맞게)
# ══════════════════════════════════════════════════════════════════════
FILES = {
    "뷰티":   "medicube.csv",
    "게임기": "nintendo.csv",
    "가구":   "ikea.csv",
    "유아":   "stoke.csv",
    "자전거": "constantin.csv",
    "에어팟": "airpod.csv",
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ★수정 구역 2★ : 친절도 측정 변수용 키워드 (독립변수 1)
# ══════════════════════════════════════════════════════════════════════
KIND_POS = ["네고가능","네고가","네고ㄱㄴ","쿨거래","쿨거","채팅","문의","편하게",
            "직거래","택배","언제든","언제든지","연락주세요","연락주","감사","부탁","빠른답장","나눔"]
KIND_NEG = ["네고불가","네고사절","노네고","찔러보기","차단","환불불가","교환불가",
            "반품불가","직거래만","사절","단순변심","비매너","노쇼"]
EMOTICON_TEXT = [r"\^\^", r"\^_\^", r":\)", r";\)", r"ㅎㅎ+", r"ㅋㅋ+", r"ㅠㅠ+", r"ㅜㅜ+", "♡", "♥"]
EMOJI_PAT = re.compile(r"[\U0001F300-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF]")  # ❤️💕🙂 등 자동 포함
PUNCT_CHARS = ["~", "!", "?"]
HON = re.compile(r"(습니다|ㅂ니다|입니다|됩니다|드립니다|드려요|세요|에요|예요|이에요|"
                 r"어요|아요|해요|워요|네요|을게요|ㄹ게요|십시오|드릴게요|구요|아용|어용)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ★수정 구역 3★ : 카테고리별 주요 단어 (독립변수 2) — 제품군마다 따로
#                단어 추가/삭제 자유. 모두 실제 본문 등장 단어로 구성.
# ══════════════════════════════════════════════════════════════════════
CATEGORY_WORDS = {
    "뷰티": [  # 미용/효과 특색 (저빈도 효과어 제외 버전)
        "피부","탄력","개선","광채","모공","효과","콜라겐","케어","관리","도움",
        "진정","톤업","트러블",
        "메디큐브","부스터","프로","에이지알","에이지","부스터프로","기기","디바이스",
        "본체","모드","충전","충전기","케이블","설명서","에디션","미니","핑크","블랙",
        "박스","개봉","정품","소독","보관",
    ],
    "게임기": [
        "스위치","닌텐도","본체","조이콘","게임","박스","구성","케이스","충전기","필름",
        "세트","케이블","그립","스트랩","컨트롤러","파우치","어댑터","독","HDMI","보호필름",
        "OLED","휴대","에디션","모델","정품","정발","칩","타이틀",
        "액정","기스","작동","정상","초기화","미개봉","풀박","흠집","깨끗","충전",
    ],
    "가구": [
        "트롤리","바퀴","이동","이동식","편리","수납","공간","정리","활용","보관",
        "높이","소재","스틸","철제",
        "주방","욕실","거실","공부방","생활","대면","제작",
        "화이트","흰색","색상","깔끔","다양","문고리","로스코그","사이즈","중고","오염",
        "하자","깨끗",
    ],
    "유아": [
        "아기","신생아","육아","아이","여아","남아","둘째","첫째","우리아이",
        "스토케","트립트랩","뉴본","세트","커버","모빌","글라이더","의자","시트",
        "안전벨트","트레이","쿠션",
        "안전","편안","세탁","오염","문고리","연장","그레이","보관","깨끗","흠집",
    ],
    "자전거": [
        "안장","스템","프레임","핸들","카본","크랭크","브레이크","체인","시마노","드롭",
        "휠","기어","변속","페달","타이어","바퀴","포크",
        "콘스탄틴","어베인","픽시","언노운","트랙","벨로시닷","펠트","위아위스","로드",
        "카본프레임","순정",
        "사이즈","대차","구성","신품","기스","하자","정비","점검","주행","교체",
        "인치","알루미늄","풀카본",
    ],
    "에어팟": [
        "에어팟","에어팟4","노이즈","노캔","액티브","ANC","세대","모델","버전","애플",
        "정품","이어폰","MXP","사운드","음질","몰입","소음","주변음",
        "케이스","박스","충전","충전기","USB","케이블","이어팁","보증","애플케어",
        "개봉","미개봉","기스","깨끗","생활기스","흠집","새것","풀박","정상",
    ],
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  이하 분석 로직 (수정 불필요)
# ══════════════════════════════════════════════════════════════════════
def cnt(text, terms):
    t = re.sub(r"\s+", "", text)
    return sum(t.count(re.sub(r"\s+", "", w)) for w in terms)
def cnt_re(text, pats):
    return sum(len(re.findall(p, text)) for p in pats)

In [ ]:
SEVEN = ["이모티콘문장부호","본문길이","사진개수","친절감소","높임어미","정보밀도","친절상승"]
SEVEN_NAME = {"이모티콘문장부호":"이모티콘/문장부호","본문길이":"본문길이","사진개수":"사진개수",
              "친절감소":"친절감소표현","높임어미":"높임어미","정보밀도":"정보밀도","친절상승":"친절상승표현"}

In [ ]:
def featurize(df, catname):
    c = df["content"].astype(str); L = c.str.len().replace(0, np.nan)
    v = pd.to_numeric(df["view"], errors="coerce")
    f = pd.DataFrame()
    f["interest_rate"] = pd.to_numeric(df["interest"], errors="coerce") / v   # 찜전환율
    f["chat_rate"]     = pd.to_numeric(df["heart"], errors="coerce") / v       # 채팅전환율
    # 독립변수1: 친절도 7요소
    emo = c.apply(lambda x: cnt_re(x, EMOTICON_TEXT) + len(EMOJI_PAT.findall(x)))
    pun = c.apply(lambda x: sum(x.count(k) for k in PUNCT_CHARS))
    f["이모티콘문장부호"] = 100 * (emo + pun) / L
    f["본문길이"] = np.log1p(L)
    f["사진개수"] = pd.to_numeric(df["image"], errors="coerce")
    f["친절감소"] = c.apply(lambda x: cnt(x, KIND_NEG))
    f["높임어미"] = c.apply(lambda x: len(HON.findall(x)))
    f["정보밀도"] = (c.str.count(r"\d") + c.str.count(r"[ \t]") + c.str.count(r"\n")) / L
    f["친절상승"] = c.apply(lambda x: cnt(x, KIND_POS))
    # 독립변수2: 카테고리별 주요 단어
    f["주요단어"] = c.apply(lambda x: cnt(x, CATEGORY_WORDS[catname]))
    f["catname"] = catname
    return f.replace([np.inf, -np.inf], np.nan)

In [ ]:
base = pd.concat([featurize(pd.read_csv(fn, encoding="utf-8-sig"), cat)
                  for cat, fn in FILES.items()], ignore_index=True)
base = base.dropna(subset=["interest_rate","chat_rate","본문길이"])

In [ ]:
def z(s):
    sd = s.std(ddof=0); return (s - s.mean()) / sd if sd else s * 0

In [ ]:
print("="*64)
print(f"분석 표본: 총 {len(base)}건")
print(base.groupby("catname").size().reindex(FILES.keys()).to_string())
print("="*64)

In [ ]:
# ── [독립변수1] 친절도 7요소 → 통합 모델 (제품군 더미 통제) ──────────────
print("\n\n████ 독립변수1: 친절도 7요소 (통합 모델, 제품군 더미 통제) ████")
for dv, lab in [("interest_rate","찜전환율"), ("chat_rate","채팅전환율")]:
    d = base.dropna(subset=[dv] + SEVEN)
    X = pd.DataFrame({v: z(d[v]) for v in SEVEN})
    dm = pd.get_dummies(d["catname"], prefix="cat", drop_first=True).astype(float)
    X = sm.add_constant(pd.concat([X.reset_index(drop=True), dm.reset_index(drop=True)], axis=1))
    m = sm.OLS(d[dv].reset_index(drop=True).astype(float), X.astype(float)).fit(cov_type="HC3")
    res = pd.DataFrame({"변수":[SEVEN_NAME[v] for v in SEVEN],
                        "coef":[round(m.params[v],5) for v in SEVEN],
                        "p":[round(m.pvalues[v],4) for v in SEVEN]})
    res["유의"] = res["p"].apply(lambda p:"***" if p<.001 else "**" if p<.01 else "*" if p<.05 else "")
    print(f"\n── DV={lab}  (n={int(m.nobs)}, R²={m.rsquared:.3f}) ──")
    print(res.to_string(index=False))

In [ ]:
# ── [독립변수2] 카테고리 주요 단어 → 제품군별 개별 회귀 ─────────────────
#    각 모델 = 주요단어 + 친절도7요소(통제)  → 주요단어의 순효과
print("\n\n████ 독립변수2: 카테고리별 주요 단어 (제품군별 개별 회귀) ████")
summary = []
for cat in FILES:
    sub = base[base.catname == cat]
    for dv, lab in [("interest_rate","찜전환율"), ("chat_rate","채팅전환율")]:
        xs = ["주요단어"] + [v for v in SEVEN if sub[v].dropna().nunique() > 1]
        d = sub.dropna(subset=[dv] + xs)
        X = sm.add_constant(pd.DataFrame({v: z(d[v]) for v in xs}))
        m = sm.OLS(d[dv].astype(float), X.astype(float)).fit(cov_type="HC3")
        co, p = m.params["주요단어"], m.pvalues["주요단어"]
        mean_dv = d[dv].mean()
        pct = co / mean_dv * 100 if mean_dv else np.nan
        sd_words = sub["주요단어"].std(ddof=0)
        summary.append({"제품군":cat, "DV":lab, "주요단어_coef":round(co,5), "p":round(p,4),
                        "평균대비%":round(pct,1), "1SD(단어수)":round(sd_words,1),
                        "유의":"O" if p<.05 else "", "n":int(m.nobs), "R2":round(m.rsquared,3)})
sumdf = pd.DataFrame(summary)
print(sumdf.to_string(index=False))

In [ ]:
# 저장
import os; os.makedirs("/mnt/user-data/outputs", exist_ok=True)
sumdf.to_csv("/mnt/user-data/outputs/unified_category_words.csv", index=False, encoding="utf-8-sig")
print("\n저장 → /mnt/user-data/outputs/unified_category_words.csv")